# MCP server 1 — Utilities

A fast, dependency-free first success that teaches the MCP request/response pattern.

**Tutorial contract:** run cells from top to bottom. Every external dependency is checked before use,
outputs go under `artifacts/kdd_tutorial/`, and no credential value is printed.


## Goal

Discover and call deterministic utility tools.

**Requires:** nothing beyond `uv sync`


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import json, os, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ENV_FILE = REPO / ".env"
if not ENV_FILE.exists():
    raise RuntimeError(f"Missing {ENV_FILE}. Complete 00_environment_setup.ipynb first.")
load_dotenv(ENV_FILE, override=True)
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("environment source:", ENV_FILE)
print("python:", sys.version.split()[0])


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

ENTRY_POINTS = {
    "iot": "iot-mcp-server", "utilities": "utilities-mcp-server",
    "fmsr": "fmsr-mcp-server", "wo": "wo-mcp-server",
    "tsfm": "tsfm-mcp-server", "vibration": "vibration-mcp-server",
}

async def mcp_session(server, operation, tool_name=None, arguments=None):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), ENTRY_POINTS[server]],
        cwd=str(REPO),
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if operation == "list":
                return await session.list_tools()
            return await session.call_tool(tool_name, arguments or {})

def text_result(result):
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text

async def list_tools(server):
    response = await mcp_session(server, "list")
    return [{"name": t.name, "description": t.description, "schema": t.inputSchema} for t in response.tools]

async def call_tool(server, name, **arguments):
    return text_result(await mcp_session(server, "call", name, arguments))


## 1. Discover the live MCP contract

This starts the real stdio server and asks it for its tool schemas.


In [ ]:
tools = await list_tools("utilities")
[(t["name"], list(t["schema"].get("properties", {}))) for t in tools]


## 2. Call two tools


In [ ]:
now = await call_tool("utilities", "current_date_time")
english = await call_tool("utilities", "current_time_english")
now, english


## 3. Read a JSON artifact through the server


In [ ]:
sample = ARTIFACTS / "utility_sample.json"
sample.write_text(json.dumps({"tutorial": "KDD", "ready": True}), encoding="utf-8")
await call_tool("utilities", "json_reader", file_name=str(sample))


## Takeaway

You exercised the server through MCP JSON-RPC over stdio—the same boundary the agents use.
